## Step 1: Dataset: Tiny Shakespeare
There's a famous cleaned version of Shakespeare that's been the standard LLM toy dataset since Andrej Karpathy's char-RNN days. It's ~1MB, plain text, perfect for a small model.

In [ ]:
import requests

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = requests.get(url)

with open("shakespeare.txt", "w") as f:
    f.write(response.text)

# Quick sanity check
text = response.text
print(f"Total characters: {len(text):,}")
print(f"First 300 chars:\n{text[:300]}")

Total characters: 1,115,394
First 300 chars:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


## Step 2: Tokenizer: Character-Level vs Subword
A MiniModel was built with a simple vocabulary. Shakespeare gives two clean options:

- Character-level — vocab is just the ~65 unique characters in the text (a-z, A-Z, punctuation, space, newline). Simple, tiny vocab, works well with a small model.
- Subword (BPE) — you'd use tiktoken or sentencepiece. Richer tokens, but vocab size jumps to thousands. Your small model would struggle with a large vocab — the output head (d_model → vocab_size) becomes massive relative to everything else.

In [ ]:
# Build character-level vocabulary
chars = sorted(set(text))
vocab_size = len(chars)
print(f"Vocab size: {vocab_size}")  # should be ~65

# Encoder: char → int
stoi = {ch: i for i, ch in enumerate(chars)}

# Decoder: int → char
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# Test it
print(encode("Hello"))       # → [20, 43, 50, 50, 53]
print(decode([20, 43, 50, 50, 53]))  # → "Hello"

# Encode the full dataset
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(f"Data shape: {data.shape}")  # → torch.Size([1115394])

Vocab size: 65
[20, 43, 50, 50, 53]
Hello
Data shape: torch.Size([1115394])


## Step 3: Train/Val Split and Hyperparameters

In [ ]:
import torch
import torch.nn as nn
import math
import requests

In [ ]:
# 90% train, 10% validation
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]
print(f"Train tokens: {len(train_data):,}")
print(f"Val tokens  : {len(val_data):,}\n")

# Hyperparameters — tuned for a small model on Colab free tier
block_size = 64    # context window — T in (B, T)
batch_size = 32     # sequences per batch — B in (B, T)
d_model    = 128    # embedding dimension
n_heads    = 4      # attention heads  (d_model must be divisible by n_heads)
d_ff       = 512   # FFN inner dimension (4 × d_model is the standard)
n_layers   = 4      # stacked transformer blocks
dropout    = 0.1    # 10% of activations zeroed during training
lr         = 3e-4   # Adam learning rate
max_iters  = 5000   # training steps
eval_every = 200    # print val loss every N steps

device     = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Training on: {device}\n")

Train tokens: 1,003,854
Val tokens  : 111,540

Training on: cpu



- `block_size = 64` — the context window. Each training example is 64 characters long. The model sees 64 characters and predicts the next character at every position. This is T (seq_len) in your shape notation.
- `batch_size = 32` — 32 independent sequences processed simultaneously. This is B in your shapes. Higher = faster training but more GPU memory.
- `d_model = 128` — embedding dimension. Every token becomes a 128-dimensional vector. Smaller than GPT-2's 768 but fine for a character-level toy model.
- `n_heads = 4` — 4 attention heads. Each head gets d_model / n_heads = 128 / 4 = 32 dimensions. Important: d_model must be divisible by n_heads.
- `n_layers = 4` — 4 stacked transformer blocks. Each block refines the representations further.
- `dropout = 0.1` — randomly zeroes 10% of activations during training. Prevents overfitting by stopping the model from relying too heavily on any single pathway. Disabled during generation.
- `lr = 3e-4` — learning rate of 0.0003. This is the classic Adam learning rate for transformers — Karpathy calls it "the best learning rate." Not too fast (explodes), not too slow (crawls).
- `max_iters = 3000` — run 3000 training steps. Each step processes one batch of 32 sequences. Enough to see clear improvement on Colab's free T4 GPU in a few minutes.

## Step 4: The Batch Loader

In [ ]:
def get_batch(split):
    # Step 1: pick which river to fish from
    data_split = train_data if split == 'train' else val_data

    # Step 2: throw 32 random darts
    # Upper limit is len-block_size so we never fall off the end
    ix = torch.randint(len(data_split) - block_size, (batch_size,))
    # ix is just 32 random integers, e.g. [40231, 812445, 3891, ...]

    # Step 3: for each dart position, grab a 64-character window
    x = torch.stack([data_split[i   : i +  block_size   ] for i in ix])
    y = torch.stack([data_split[i+1 : i +  block_size+1 ] for i in ix])
    # x shape: (32, 64) — inputs
    # y shape: (32, 64) — targets (same windows, shifted right by 1)

    return x.to(device), y.to(device)

# Test it
xb, yb = get_batch('train')
print(f"Input shape:  {xb.shape}")   # → (32, 64)
print(f"Target shape: {yb.shape}")   # → (32, 64)

Input shape:  torch.Size([32, 64])
Target shape: torch.Size([32, 64])


## Step 5: Positional Encoding

In [ ]:
def positional_encoding(seq_len, d_model, device):
    """
    Returns PE matrix of shape (seq_len, d_model).
    Same sinusoidal formula as before — now takes device as arg
    so it lands on the right hardware automatically.
    """
    pe = torch.zeros(seq_len, d_model, device=device)
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            denom        = 10000 ** (i / d_model)
            pe[pos, i]   = math.sin(pos / denom)
            if i + 1 < d_model:
                pe[pos, i+1] = math.cos(pos / denom)
    return pe

## **Step 6: Multi Head Attention - Batch Aware**

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model   = d_model
        self.num_heads = num_heads
        self.d_k       = d_model // num_heads

        self.W_Q     = nn.Linear(d_model, d_model, bias=False)
        self.W_K     = nn.Linear(d_model, d_model, bias=False)
        self.W_V     = nn.Linear(d_model, d_model, bias=False)
        self.W_O     = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, kv_cache=None):
        """
        x shape:
          Training / prefill : (B, T, d_model)   — full sequence
          Decode (1 new token): (B, 1, d_model)

        kv_cache: dict with 'K' and 'V' of shape (B, num_heads, T_past, d_k)
                  or None on the first call.

        Returns:
          output   : same shape as x
          new_kv_cache : updated cache dict
        """
        B, T, C = x.shape                  # B=batch, T=seq_len this call, C=d_model

        # ── Project to Q, K, V ──────────────────────────────────────────────
        Q = self.W_Q(x)                    # (B, T, d_model)
        K = self.W_K(x)                    # (B, T, d_model)
        V = self.W_V(x)                    # (B, T, d_model)

        # ── Split into heads ────────────────────────────────────────────────
        # (B, T, d_model) → (B, T, num_heads, d_k) → (B, num_heads, T, d_k)
        Q = Q.view(B, T,  self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(B, T,  self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(B, T,  self.num_heads, self.d_k).transpose(1, 2)
        # Q shape: (B, num_heads, T, d_k)
        # K shape: (B, num_heads, T, d_k)  — will grow after cache concat

        # ── KV Cache ────────────────────────────────────────────────────────
        if kv_cache is not None:
            # Append new K, V to cached K, V along the sequence axis (dim=2)
            # Cached K shape: (B, num_heads, T_past, d_k)
            # New K shape:    (B, num_heads, T,      d_k)
            # After concat:   (B, num_heads, T_past + T, d_k)
            K = torch.cat([kv_cache['K'], K], dim=2)
            V = torch.cat([kv_cache['V'], V], dim=2)

        new_kv_cache = {'K': K, 'V': V}

        full_len = K.shape[2]              # total key/value length (past + new)

        # ── Attention Scores ─────────────────────────────────────────────────
        # Q: (B, heads, T, d_k)  ×  Kᵀ: (B, heads, d_k, full_len)
        # → scores: (B, heads, T, full_len)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)

        # ── Causal Mask ──────────────────────────────────────────────────────
        # Only when processing multiple tokens (training + prefill).
        # During decode T == 1: the single new token can attend to all cached
        # past tokens freely — no masking needed.
        if T > 1:
            # mask shape: (T, full_len) — broadcasts over (B, heads) automatically
            mask = torch.triu(
                torch.ones(T, full_len, device=x.device),
                diagonal=full_len - T + 1
            ).bool()
            scores = scores.masked_fill(mask, float('-inf'))

        # ── Softmax + Weighted Sum ───────────────────────────────────────────
        attn_weights = torch.softmax(scores, dim=-1)         # (B, heads, T, full_len)
        attn_weights = self.dropout(attn_weights)            # dropout on attention weights
        out = attn_weights @ V                               # (B, heads, T, d_k)

        # ── Merge heads back ────────────────────────────────────────────────
        # (B, heads, T, d_k) → (B, T, heads, d_k) → (B, T, d_model)
        out = out.transpose(1, 2).contiguous().view(B, T, C)

        return self.W_O(out), new_kv_cache

## **Step 7: Feed Forward Network (FFN)**

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)  # works on any shape (..., d_model)

## Step 8: The Transformer Block

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.attention    = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1        = nn.LayerNorm(d_model)
        self.norm2        = nn.LayerNorm(d_model)

    def forward(self, x, kv_cache=None):
        # Pre-LN: normalize BEFORE feeding into sublayer
        attn_out, new_kv_cache = self.attention(self.norm1(x), kv_cache=kv_cache)
        x = x + attn_out                  # residual connection

        ff_out = self.feed_forward(self.norm2(x))
        x = x + ff_out                    # residual connection

        return x, new_kv_cache

## **Step 9: The Full MiniGPT Model**

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, dropout):
        super().__init__()
        self.embedding   = nn.Embedding(vocab_size, d_model)
        self.emb_dropout = nn.Dropout(dropout)
        self.blocks      = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.final_norm  = nn.LayerNorm(d_model)
        self.output_head = nn.Linear(d_model, vocab_size)

    def forward(self, token_ids, kv_caches=None, start_pos=0):
        """
        token_ids: (B, T)  during training
                   (B, T)  during prefill  (T = prompt length)
                   (B, 1)  during decode   (one new token per step)

        kv_caches: list of num_layers cache dicts, or None
        start_pos: where in the full sequence does this input begin
                   (0 for training/prefill, grows during decode)

        Returns:
            logits     : (B, T, vocab_size)
            kv_caches  : updated list of per-block caches
        """
        B, T = token_ids.shape

        if kv_caches is None:
            kv_caches = [None] * len(self.blocks)

        # Step 1: Embed tokens → (B, T, d_model)
        x = self.embedding(token_ids)
        x = self.emb_dropout(x)

        # Step 2: Add positional encoding
        # Build PE for all positions up to start_pos + T, then slice the relevant rows
        pe               = positional_encoding(start_pos + T, self.embedding.embedding_dim, token_ids.device)
        pe_slice         = pe[start_pos : start_pos + T]    # (T, d_model)
        x                = x + pe_slice                     # broadcasts over batch dim

        # Step 3: Pass through transformer blocks
        new_kv_caches = []
        for block, block_cache in zip(self.blocks, kv_caches):
            x, new_block_cache = block(x, kv_cache=block_cache)
            new_kv_caches.append(new_block_cache)

        # Step 4: Final norm + output head
        x      = self.final_norm(x)
        logits = self.output_head(x)      # (B, T, vocab_size)

        return logits, new_kv_caches

## **Step 10: Instantiate Model**

In [ ]:
model = MiniGPT(
    vocab_size = vocab_size,
    d_model    = d_model,
    num_heads  = n_heads,
    d_ff       = d_ff,
    num_layers = n_layers,
    dropout    = dropout,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}\n")

Model parameters: 808,001



## **Step 11: The Training Loop**

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
loss_fn   = nn.CrossEntropyLoss()

@torch.no_grad()
def estimate_val_loss(eval_batches=20):
    """
    Run eval_batches batches through the model in eval mode
    and return the average validation loss.
    """
    model.eval()
    total_loss = 0.0
    for _ in range(eval_batches):
        xb, yb     = get_batch('val')
        logits, _  = model(xb)
        # logits: (B, T, vocab_size) → (B*T, vocab_size)
        # yb:     (B, T)             → (B*T,)
        loss       = loss_fn(logits.view(-1, vocab_size), yb.view(-1))
        total_loss += loss.item()
    model.train()
    return total_loss / eval_batches

print("Starting training...\n")

for step in range(max_iters):

    # ── Get a fresh random batch ────────────────────────────
    xb, yb = get_batch('train')           # xb: (B, T),  yb: (B, T)

    # ── Forward pass ────────────────────────────────────────
    logits, _ = model(xb)                 # logits: (B, T, vocab_size)

    # ── Compute loss ─────────────────────────────────────────
    # CrossEntropyLoss expects:
    #   input : (N, vocab_size)   — one row per prediction
    #   target: (N,)              — one int per prediction
    # So we flatten (B, T, vocab_size) → (B*T, vocab_size)
    #          and  (B, T)             → (B*T,)
    loss = loss_fn(
        logits.view(-1, vocab_size),      # (B*T, vocab_size)
        yb.view(-1)                       # (B*T,)
    )

    # ── Backward pass ────────────────────────────────────────
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # ── Logging ──────────────────────────────────────────────
    if step % eval_every == 0 or step == max_iters - 1:
        val_loss = estimate_val_loss()
        print(f"Step {step:4d} | Train loss: {loss.item():.4f} | Val loss: {val_loss:.4f}")

print("\nTraining complete.")

Starting training...

Step    0 | Train loss: 4.3277 | Val loss: 4.1157
Step  200 | Train loss: 2.4759 | Val loss: 2.4590
Step  400 | Train loss: 2.3509 | Val loss: 2.2776
Step  600 | Train loss: 2.2128 | Val loss: 2.1743
Step  800 | Train loss: 2.1447 | Val loss: 2.1280
Step 1000 | Train loss: 2.0394 | Val loss: 2.0672
Step 1200 | Train loss: 1.9734 | Val loss: 2.0056
Step 1400 | Train loss: 1.9212 | Val loss: 1.9575
Step 1600 | Train loss: 1.9239 | Val loss: 1.9344
Step 1800 | Train loss: 1.9024 | Val loss: 1.9054
Step 2000 | Train loss: 1.7873 | Val loss: 1.8902
Step 2200 | Train loss: 1.8567 | Val loss: 1.8527
Step 2400 | Train loss: 1.7833 | Val loss: 1.8526
Step 2600 | Train loss: 1.7789 | Val loss: 1.8348
Step 2800 | Train loss: 1.7192 | Val loss: 1.8229
Step 3000 | Train loss: 1.9392 | Val loss: 1.8124
Step 3200 | Train loss: 1.7603 | Val loss: 1.8127
Step 3400 | Train loss: 1.6949 | Val loss: 1.7911
Step 3600 | Train loss: 1.6893 | Val loss: 1.7729
Step 3800 | Train loss: 1.64

## **Step 12: Save the Model Weights**

In [ ]:
torch.save({
    'model_state_dict' : model.state_dict(),
    'vocab_size'       : vocab_size,
    'd_model'          : d_model,
    'n_heads'          : n_heads,
    'd_ff'             : d_ff,
    'n_layers'         : n_layers,
    'dropout'          : dropout,
    'stoi'             : stoi,
    'itos'             : itos,
}, 'minigpt_shakespeare.pt')

print("Model saved to minigpt_shakespeare.pt")

Model saved to minigpt_shakespeare.pt


## **Step 13: The Generation Flow**

In [ ]:
# ==============================================================================
# TOP-K APPROACH
# ==============================================================================
# def _sample(logits_1d, temperature, top_k, generated=None, rep_penalty=1.3):
#     # Penalise recently seen tokens
#     if generated and rep_penalty > 1.0:
#         for token_id in set(generated[-20:]):  # look at last 20 chars
#             logits_1d[token_id] /= rep_penalty  # divide logit → lower probability

#     logits_1d = logits_1d / temperature
#     if top_k is not None:
#         values, _ = torch.topk(logits_1d, top_k)
#         logits_1d[logits_1d < values[-1]] = float('-inf')

#     probs = torch.softmax(logits_1d, dim=-1)
#     return torch.multinomial(probs, num_samples=1).item()


# ==============================================================================
# TOP-P APPROACH
# ==============================================================================

def _sample(logits_1d, temperature, top_p=0.9, generated=None, rep_penalty=1.2):
    """
    Temperature + repetition penalty + nucleus (top-p) sampling.
    """
    # Step 1: Repetition penalty — suppress recently seen characters
    if generated and rep_penalty > 1.0:
        for token_id in set(generated[-15:]):
            logits_1d[token_id] /= rep_penalty

    # Temperature = 0 → pure greedy
    if temperature == 0:
        return logits_1d.argmax().item()

    # Step 2: Temperature scaling
    logits_1d = logits_1d / temperature

    # Step 3: Convert to probabilities
    probs = torch.softmax(logits_1d, dim=-1)

    # Step 4: Top-p nucleus sampling
    # Sort probabilities descending
    sorted_probs, sorted_indices = torch.sort(probs, descending=True)

    # Compute cumulative probabilities
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

    # Find cutoff: remove tokens once cumulative prob exceeds p
    # shift by 1 so we always keep at least the top token
    sorted_indices_to_remove = cumulative_probs - sorted_probs > top_p

    # Zero out the removed tokens
    sorted_probs[sorted_indices_to_remove] = 0.0

    # Renormalise so remaining probs sum to 1
    sorted_probs = sorted_probs / sorted_probs.sum()

    # Sample from the nucleus
    sampled_idx = torch.multinomial(sorted_probs, num_samples=1)

    # Map back to original token index
    return sorted_indices[sampled_idx].item()

In [ ]:
def generate(model, prompt, max_new_tokens=200, temperature=0.6, top_p=0.9, rep_penalty=1.2):
    model.eval()
    token_ids = torch.tensor([encode(prompt)], dtype=torch.long).to(device)
    generated = token_ids[0].tolist()

    with torch.no_grad():

        # Prefill
        logits, kv_caches = model(token_ids, kv_caches=None, start_pos=0)
        next_id = _sample(logits[0, -1], temperature, top_p=top_p,
                          generated=generated, rep_penalty)
        generated.append(next_id)

        # Decode
        for _ in range(max_new_tokens - 1):
            current_pos  = len(generated) - 1
            input_tensor = torch.tensor([[next_id]], dtype=torch.long).to(device)
            logits, kv_caches = model(input_tensor, kv_caches=kv_caches,
                                      start_pos=current_pos)
            next_id = _sample(logits[0, -1], temperature, top_p=top_p,
                              generated=generated, rep_penalty)
            generated.append(next_id)

    return decode(generated)

In [ ]:
print("\n=== Sample generation ===\n")
output = generate(
    model,
    prompt         = "To be or",
    max_new_tokens = 200,
    temperature    = 0.6,
    top_p          = 0.9,
    rep_penalty    = 1.2
)
print(output)


=== Sample generation ===

To be ord,
And like as of the fair words to me,
His thy was and suchomeeerrphallllee t on mid alere me, in alerrt alallllame iontow h uck w hellallllllly minge ouck th te at te aly aten on he, y w alis ton al
